**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Compressed Sensing

Break Nyquist — legally. If a signal is *sparse* in some basis, you can reconstruct it from far fewer measurements than samples, by asking the right (random!) questions and solving the right (L1!) puzzle. Two sessions: why it works, and a full reconstruction you'll code yourself. This is the mathematics behind fast MRI.

## 1. Pre-requisites

- [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb) S1–S2 (bases, least squares).
- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) S1 (convexity).
- [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) S5 (sampling — the law we're bending).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 2 — *Why Undersampling Can Work* (~35 min)
**Goal:** sparsity + incoherent measurements + the L1 trick: the three-ingredient recipe.
**Builds on:** [Linear Algebra](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb). &nbsp; **Feeds into:** Session 2 (reconstruction lab).

---

## 2. The Three Ingredients

💡 **Intuition.** Nyquist protects you against the *worst-case* signal. But natural signals aren't worst-case — they're **sparse**: a few active coefficients in the right basis ([Linear Algebra S1](../Intro_Math/Linear_Algebra/Linear_Algebra.ipynb)'s smooth bump needed ~6 cosine coefficients). If only $K \ll N$ numbers matter, shouldn't $\sim K \log N$ *well-chosen* measurements suffice? They do, if the measurements are **incoherent** — each one touches a little of everything (random projections are perfect) — so that few measurements still 'see' every possible sparse pattern.

**The recovery puzzle.** Measurements $y = \Phi x$ with $\Phi \in \mathbb{R}^{m\times N}$, $m \ll N$: infinitely many $x$ fit. Choosing the sparsest ($\min \|x\|_0$) is combinatorial. The miracle: minimizing the **L1 norm** — convex! — finds the same answer under incoherence conditions (RIP; Candès–Romberg–Tao / Donoho, stated not proved).

💡 **Intuition.** Why L1 and not L2? Geometry. The solution set of $y = \Phi x$ is a flat (affine subspace); recovery inflates a norm-ball until it first touches the flat. The L2 ball is round — it touches at a generic point with *all* coordinates nonzero. The L1 ball is a diamond whose **corners sit on the axes** — flats almost always touch a corner first, and corners are sparse points. Sparsity falls out of the shape of the ball.

In [2]:
# The diamond-vs-ball picture, in 2-D
th = np.linspace(0, 2*np.pi, 400)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.6))
# constraint line y = Φx: x0 + 2 x1 = 1.4
for ax, name in [(axes[0], "L2"), (axes[1], "L1")]:
    xs = np.linspace(-1.6, 1.6, 10)
    ax.plot(xs, (1.4 - xs)/2, "k", linewidth=1, label="all x with Φx = y")
    if name == "L2":
        r = 1.4/np.sqrt(5)
        ax.plot(r*np.cos(th)*np.sqrt(5)/np.sqrt(5), r*np.sin(th), "C0")
        ax.plot(np.cos(th)*0.626, np.sin(th)*0.626, "C0")
        pt = np.array([1.4, 2.8])/5
    else:
        s = 0.7
        ax.plot([s,0,-s,0,s], [0,s,0,-s,0], "C1")
        pt = np.array([0, 0.7])
    ax.plot(*pt, "r*", markersize=14, label="first touch")
    ax.set_title(f"{name} ball meets the flat: {'dense point' if name=='L2' else 'CORNER → sparse!'}")
    ax.set_xlim(-1.6, 1.6); ax.set_ylim(-1.2, 1.4); ax.legend(fontsize=7); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

/tmp/ipykernel_2034816/541760034.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 2 — *Reconstruction Lab* (~40 min)
**Goal:** recover a sparse spectrum from 15% of the samples with ISTA, coded from scratch.
**Builds on:** Session 1.

---

## 3. The Lab

Scenario: a signal made of 5 tones. Nyquist says record all $N = 1024$ samples; we record **150 random ones** and recover the whole thing.

Solver: **ISTA** (iterative soft-thresholding) for the LASSO form $\min_c \tfrac12\|y - \Phi \Psi c\|^2 + \lambda \|c\|_1$ — just [gradient descent](../Intro_Math/Optimization/Optimization.ipynb) on the smooth part, plus a *shrink-toward-zero* step that enforces sparsity:
$$c \leftarrow \mathrm{soft}_{\lambda\eta}\big(c - \eta \, A^T(Ac - y)\big), \qquad \mathrm{soft}_\tau(u) = \mathrm{sign}(u)\max(|u| - \tau, 0)$$

In [3]:
N, m = 1024, 150
# sparse-in-frequency signal: 5 random tones (real DCT basis keeps everything real)
from scipy.fft import dct, idct
c_true = np.zeros(N)
support = rng.choice(N, 5, replace=False)
c_true[support] = rng.uniform(1, 3, 5) * rng.choice([-1, 1], 5)
x_true = idct(c_true, norm="ortho")

# measure: m random time samples (a random row-selector Φ — maximally incoherent with DCT)
keep = np.sort(rng.choice(N, m, replace=False))
y = x_true[keep]

def A(c):  return idct(c, norm="ortho")[keep]          # Φ Ψ c
def AT(r):
    z = np.zeros(N); z[keep] = r
    return dct(z, norm="ortho")                         # (Φ Ψ)ᵀ r

# ISTA
c = np.zeros(N); eta, lam = 1.0, 0.02
for it in range(400):
    c = c - eta * AT(A(c) - y)
    c = np.sign(c) * np.maximum(np.abs(c) - lam * eta, 0)

x_rec = idct(c, norm="ortho")
print(f"recovered support: {np.sort(np.abs(c).argsort()[-5:])}")
print(f"true support:      {np.sort(support)}")
print(f"reconstruction SNR: {10*np.log10(np.var(x_true)/np.var(x_rec - x_true)):.1f} dB from {m}/{N} = {m/N:.0%} of samples")

recovered support: [275 315 522 650 867]
true support:      [275 315 522 650 867]
reconstruction SNR: 25.8 dB from 150/1024 = 15% of samples


In [4]:
fig, axes = plt.subplots(2, 1, figsize=(9, 4), sharex=True)
axes[0].plot(x_true, linewidth=1, label="true signal (1024 samples)")
axes[0].plot(keep, y, "r.", markersize=3, label="the 150 we measured")
axes[0].legend(fontsize=8); axes[0].set_xlim(0, 400)
axes[1].plot(x_rec, linewidth=1, color="C2", label="L1 reconstruction from 15%")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

/tmp/ipykernel_2034816/3084294164.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [5]:
# Control experiment: least squares (L2) from the same 150 samples — total failure
# minimum-norm solution: c = Aᵀ(AAᵀ)⁻¹y, computed via lstsq on the explicit matrix
Psi_rows = idct(np.eye(N), norm="ortho", axis=0)[keep]     # the m×N matrix ΦΨ
c_l2, *_ = np.linalg.lstsq(Psi_rows, y, rcond=None)
x_l2 = idct(c_l2, norm="ortho")
print(f"L2 reconstruction SNR: {10*np.log10(np.var(x_true)/np.var(x_l2 - x_true)):.1f} dB   ← the round ball touching a dense point")
print(f"L1 reconstruction SNR: {10*np.log10(np.var(x_true)/np.var(x_rec - x_true)):.1f} dB   ← the diamond finding the corner")

L2 reconstruction SNR: 0.8 dB   ← the round ball touching a dense point
L1 reconstruction SNR: 25.8 dB   ← the diamond finding the corner


**Knobs to turn** (5 minutes each): drop $m$ until recovery breaks (~$2K\log N$); make the signal less sparse (25 tones); sample *uniformly* instead of randomly and watch coherent aliasing kill it — randomness is not optional.

## 4. Conclusion

Sparsity + incoherent (random) measurements + L1's cornered geometry = signals from far fewer samples than Nyquist demands. The same three ingredients accelerate MRI scanners, single-pixel cameras, and radio astronomy.

---
## Where next

- [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) — wavelets: the sparsifying basis for images.
- [Optimization](../Intro_Math/Optimization/Optimization.ipynb) — ISTA is proximal gradient descent; the theory generalizes.
- [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb) — LASSO as MAP estimation with a Laplacian prior.